In [1]:
import pandas as pd

# Paths to Metadata
file_path_2012 = "D:\DATA\glasdata 2012.xlsx"
file_path_2013 = "D:\DATA\glasdata 2013.xlsx"

# Check available sheet names
xls_2012 = pd.ExcelFile(file_path_2012)
xls_2013 = pd.ExcelFile(file_path_2013)
print("Sheet names: ")
print("2012: ", xls_2012.sheet_names)
print("2013: ", xls_2013.sheet_names)


Sheet names: 
2012:  ['Januar', 'Februar', 'Marts', 'April', 'Maj', 'Juni', 'Juli', 'August', 'September', 'Oktober', 'November', 'December', 'Ark2']
2013:  ['datafile', 'Ark1']


In [2]:
# Read all sheets from each file into dict of DataFrames
sheets_2012 = pd.read_excel(file_path_2012, sheet_name=None, header = 4)
sheets_2013 = pd.read_excel(file_path_2013, sheet_name=None, header = 0)

# Exclude unwanted sheets
exclude_sheets = {"Ark1", "Ark2"}

df_2012 = pd.concat(
    [df.assign(Sheet=sheet, Year=2012) 
     for sheet, df in sheets_2012.items() if sheet not in exclude_sheets],
    ignore_index=True
)

df_2013 = pd.concat(
    [df.assign(Sheet=sheet, Year=2013) 
     for sheet, df in sheets_2013.items() if sheet not in exclude_sheets],
    ignore_index=True
)

# Combine everything
df_all = pd.concat([df_2012, df_2013], ignore_index=True)

In [3]:
# Show head of combined df
print(df_all.head())

print("\nData frame shape: ", df_all.shape)
print("\nColumn names: \n", df_all.columns.tolist())

  serviceyder      rekvnr    modtdato                     undafd  undyder  \
0     3800Q80  12200001.0  2012-01-02  Næstved3 Patologiafdeling      NaN   
1     3800Q80  12200002.0  2012-01-02  Næstved3 Patologiafdeling      NaN   
2     3800Q80  12200010.0  2012-01-02  Næstved3 Patologiafdeling      NaN   
3     3800Q80  12200011.0  2012-01-02  Næstved3 Patologiafdeling      NaN   
4     3800Q80  12200012.0  2012-01-02  Næstved3 Patologiafdeling      NaN   

  rekvprio team sex  alder alder gruppe  ...  andcelle krommat  Lægepoint  \
0       PF  URO   M   52.0        50-54  ...       NaN     NaN        103   
1       PF  GAS   F   77.0          65+  ...       NaN     NaN         73   
2       NO  URO   F   43.0        40-44  ...       NaN     NaN         19   
3       NO  GYN   F   57.0        55-59  ...       NaN     NaN         25   
4       NO  LUG   F   55.0        55-59  ...       NaN     NaN         19   

  bioanalytikerpoint sekretærpoint laboratoriebetjentpoint  Molekylærbiolo

In [4]:
# Select relevant colums
selected_columns = ['rekvnr', 'modtdato', 'team', 'sex', 'alder', 'alder gruppe', 'rekvdato', 'matantal', 'mattype', 'mattype tekst', 'makrotekst', 'mikrotekst', 'snomed kode', 'kode fritekst']

# Create a new DataFrame with only selected columns
df_selected = df_all[selected_columns].copy()

print(df_selected.head())

print("\nData frame shape: ", df_selected.shape)

       rekvnr    modtdato team sex  alder alder gruppe    rekvdato  matantal  \
0  12200001.0  2012-01-02  URO   M   52.0        50-54  2011-12-30      12.0   
1  12200002.0  2012-01-02  GAS   F   77.0          65+  2011-12-30       1.0   
2  12200010.0  2012-01-02  URO   F   43.0        40-44  2011-12-29       1.0   
3  12200011.0  2012-01-02  GYN   F   57.0        55-59  2011-12-30       2.0   
4  12200012.0  2012-01-02  LUG   F   55.0        55-59  2011-12-29       1.0   

   mattype mattype tekst                                         makrotekst  \
0     11.0     Hist. små  Trådformet vævsstykke, 15 mm. 1 kps. /PWN,CN ;...   
1     11.0     Hist. små  2 trådformede vævsstykker, hhv. 12 og 14 mm. 1...   
2     11.0     Hist. små  Vævsstykke, 15 x 10 x 5 mm. Flækkes, halvdelen...   
3     11.0     Hist. små  Mucinøst skrab, 4 x 3 x 1 mm. 1 kps. /PWN,CN ;...   
4     11.0     Hist. små     skrab målende 25x25x3 mm, alt med i 1 kp /PWN    

                                          mi

In [5]:
# Overview of all missing values
missing_counts = df_selected.isnull().sum()
print("\nMissing values: \n", missing_counts)

# Rows with missing rekvnr
missing_rekvnr = df_selected[df_selected["rekvnr"].isnull()]
print("\nRows with missing rekvnr: \n",missing_rekvnr)


Missing values: 
 rekvnr              24
modtdato            24
team                24
sex                 24
alder               24
alder gruppe        24
rekvdato            24
matantal            24
mattype             24
mattype tekst       24
makrotekst       67434
mikrotekst       63519
snomed kode         26
kode fritekst       25
dtype: int64

Rows with missing rekvnr: 
        rekvnr modtdato team  sex  alder alder gruppe rekvdato  matantal  \
3222      NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
3223      NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
6473      NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
6474      NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
10047     NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
10048     NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
12897     NaN      NaN  NaN  NaN    NaN          NaN      NaN       NaN   
12898     NaN    

In [6]:
# Drop rows where "rekvnr" is missing (NaN in all columns)
df_clean = df_selected.dropna(subset=["rekvnr"]).copy()

print("Data frame shape: ", df_clean.shape)

Data frame shape:  (106305, 14)


In [7]:
# Check Missing Counts Again
missing_counts = df_clean.isnull().sum()
print("\nMissing values: \n", missing_counts)


Missing values: 
 rekvnr               0
modtdato             0
team                 0
sex                  0
alder                0
alder gruppe         0
rekvdato             0
matantal             0
mattype              0
mattype tekst        0
makrotekst       67410
mikrotekst       63495
snomed kode          2
kode fritekst        1
dtype: int64


In [8]:
# Duplicate Rows
print("Number of duplicated rows:", df_clean.duplicated().sum())

Number of duplicated rows: 2842


In [ ]:
# Remove rows that are fully duplicated
df_no_dups = df_clean.drop_duplicates()
print("Number of duplicated rows:", df_no_dups.duplicated().sum())

print("Original shape:", df_clean.shape)
print("After removing duplicated rows:", df_no_dups.shape)


In [ ]:
# Save to Excel
output_file = "D:\DATA\initial_cleaning.xlsx"
df_no_dups.to_excel(output_file, index=False)

print(f"Saved DataFrame to {output_file}")